# eph_04 — Spatial encoding: where do RT and kinematics encoders cluster in CCF?

Projects per-unit encoding T-statistics onto Allen Common Coordinate Framework (CCF)
brain coordinates and tests for anatomical clustering.

**Pipeline:**
1. Re-fit RT (OLS) and kinematic (Spearman) encoding using `fit_encoding`
2. Register results in `PerUnitStatsRegistry`
3. Build `SpatialEncoder(filtered_ephys, mesh_path)` — joins T-stats to CCF coords
4. 3-panel CCF plots (sagittal / horizontal / coronal)
5. Subgroup maps: neg-sig vs pos-sig spatial separation
6. Permutation test: does T-stat magnitude vary with CCF position?

## 1. Setup

In [ ]:
%matplotlib inline
import contextlib, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_04_spatial"
SAVE_FIG = False
print(f"ENV={ENV}  FOR_LOCAL={FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)
    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)
if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")
print("all_counts_df:", all_counts_df.shape)

## 3. Imports

In [ ]:
from encoding_methods import AnalysisSpec, fit_encoding
from per_unit_stats_registry import PerUnitStatsRegistry
from spatial_encoding import SpatialEncoder, spatial_dependence_summary
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

## 4. Fit encoding and register

Re-fit RT (OLS) and kinematic (Spearman) analyses. Cheap to re-compute;
no saved results needed.

In [ ]:
# Fit the same RT analysis as eph_01 so T-stats are available in this notebook.
# We re-fit rather than loading a saved result because fit_encoding is cheap.
RT_SPEC = AnalysisSpec(
    name="ols_rt",
    predictor_col="reaction_time_firstmove",
    response_col="spike_count",
    method="ols",
    trial_query="reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.5",
    log_x=True,
    zscore_x=True,
    notes="ols: spike_count ~ log(RT) — same as eph_01",
)
with contextlib.redirect_stdout(io.StringIO()):
    rt_result = fit_encoding(all_counts_df, RT_SPEC)
print(f"ols_rt n_sig: {rt_result.n_sig()}")

In [ ]:
reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)
reg.register(rt_result)

# Kinematics predictors (same as eph_02)
KIN_PREDICTORS = {
    "endpoint_x":  "first_move_endpoint_x",
    "endpoint_y":  "first_move_endpoint_y",
    "peak_vel":    "first_move_peak_velocity",
    "mean_vel":    "first_move_out_mean_velocity",
    "duration":    "first_move_out_duration",
    "distance":    "first_move_out_total_distance",
}
KIN_QUERY = "reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.0"
all_counts_df["first_move_excursion_angle_deg_abs"] = (
    all_counts_df["first_move_excursion_angle_deg"].abs()
)
kin_specs = [
    AnalysisSpec(
        name=f"spearman_{key}",
        predictor_col="spike_count",
        response_col=col,
        method="spearman",
        trial_query=KIN_QUERY,
    )
    for key, col in KIN_PREDICTORS.items()
]
with contextlib.redirect_stdout(io.StringIO()):
    for spec in kin_specs:
        reg.register(fit_encoding(all_counts_df, spec))

print(reg)

## 5. Build SpatialEncoder

`SpatialEncoder` loads the LC mesh, converts vertices to bregma-centered LPS mm,
and stores contour outlines for each anatomical plane.

In [ ]:
# Build SpatialEncoder from filtered_ephys (which has x_ccf, y_ccf, z_ccf)
if ENV == "codeocean":
    MESH_PATH = str(DATA_ROOT / "LC-NE_scratch_data_1" / "combined" / "ccf_maps" /
                    "20250418_transformed_remesh_10_ccf25.obj")
else:
    MESH_PATH = str(FOR_LOCAL / "20250418_transformed_remesh_10_ccf25.obj")

enc = SpatialEncoder(filtered_ephys, mesh_path=MESH_PATH, fold_left=True)
print("SpatialEncoder ready.  Contour planes:", list(enc._contours.keys()))

## 6. CCF map — RT encoding

Color each unit by its T-statistic for the spike_count ~ log(RT) regression.

In [ ]:
# 3-panel CCF map: spike_count ~ log(RT) t-stats
# Signed: cool = neg (faster RT with more spikes), warm = pos (slower RT)
fig, axes = enc.plot(
    reg, "ols_rt",
    use_sig=False,
    title="RT encoding — signed t-stat (spike_count ~ log RT)",
)
save_fig(fig, "ccf_rt_signed", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

fig, axes = enc.plot(
    reg, "ols_rt",
    abs_value=True, use_sig=False,
    title="RT encoding — |t-stat| (spike_count ~ log RT)",
)
save_fig(fig, "ccf_rt_abs", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 7. Subgroup map — neg vs pos RT encoders

In [ ]:
# Subgroup map: neg- vs pos-significant units; tests for spatial clustering
fig, axes, clust_results = enc.plot_subgroups(
    reg, "ols_rt",
    sig_only=True,
    title="RT encoding — signed subgroups",
)
print("Clustering test results:")
for grp, r in clust_results.items():
    print(f"  {grp}: n={r['n']}, NND={r['mean_nnd']:.3f}mm, p_clust={r['p_clustering']:.4f}")
save_fig(fig, "ccf_rt_subgroups", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 8. Spatial dependence permutation test

Two permutation tests:
- **Linear trend**: R² from value ~ x + y + z, compared to null distribution
- **kNN CV**: cross-validated kNN predictability, compared to null

Uses 500 permutations locally (fast), 5000 on Code Ocean (publication quality).

In [ ]:
# Formal permutation test: does RT encoding strength vary with CCF position?
# Fast: 500 perms for exploration; bump to 5000 for final.
PERMS = 500 if ENV == "local" else 5000

stat_rt = enc.test(reg, "ols_rt", permutations=PERMS, k_neighbors=10, also_abs=True)

print("RT spatial dependence (signed):")
for k, v in stat_rt["signed"].items():
    if k != "n_used":
        print(f"  {k}: {v}")
print("RT spatial dependence (|t|):")
for k, v in stat_rt["abs"].items():
    if k != "n_used":
        print(f"  {k}: {v}")

## 9. CCF maps — kinematic predictors

In [ ]:
# CCF maps for kinematic predictors
for spec in kin_specs:
    try:
        fig, _ = enc.plot(
            reg, spec.name,
            use_sig=False,
            title=f"Kinematic encoding — {spec.name}",
        )
        save_fig(fig, f"ccf_{spec.name}", fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()
    except Exception as e:
        print(f"Skipping {spec.name}: {e}")

## 10. Overlap map — which units encode both RT and kinematics?

In [ ]:
# Compare RT vs kinematic CCF distributions — which units are sig for both?
# Example: RT vs endpoint_y
try:
    merged = reg.compare("ols_rt", "spearman_endpoint_y")
    fig, _ = enc.plot_compare(
        reg, "ols_rt", "spearman_endpoint_y",
        merged=merged,
        title="RT vs endpoint_y — significance overlap on CCF",
    )
    save_fig(fig, "ccf_compare_rt_endpoint_y", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()
except Exception as e:
    print(f"compare failed: {e}")